In [ ]:
import pandas as pd
import numpy as np

tvad_anno = pd.read_csv('tvad_anno.csv')
tvad_shot_info = pd.read_csv('tvad_shot_info.csv')
tvad_anno_context = pd.read_csv('tvad_anno_context-9-24_face-0.2-0.4.csv')
tvad_anno_context_st = pd.read_csv('tvad_anno_context-9-24_face-0.2-0.4_scale_thread.csv')

In [ ]:
stage1 = pd.read_csv("stage1_qwen2vl.csv")

In [3]:
PLACEHOLDER = 7.7 

def build_phase_a(stage1: pd.DataFrame) -> pd.DataFrame:
    df = stage1.copy()

    scene_id = "s" + df["anno_idx"].astype(str)
    cand_id = "A"

    scene_dur = df["end"] - df["start"]
    speech_dur = PLACEHOLDER
    silence_dur = scene_dur - speech_dur

    n_silence_windows = PLACEHOLDER 
    n_visual_events = 5 

    n_tokens = df["text_gen"].apply(lambda t: len(str(t).split()))

    TTS_speech_dur = n_tokens / 2.5 
    fits_silence = (TTS_speech_dur <= silence_dur).astype(int)

    human_AD_sim = PLACEHOLDER
    cov_score = PLACEHOLDER
    redun_score = PLACEHOLDER

    quality_score = (
        0.5 * human_AD_sim
        + 0.3 * (cov_score / 5)
        + 0.2 * (1 - (redun_score / 5))
    )

    phaseA = pd.DataFrame({
        "scene_id": scene_id,
        "cand_id": cand_id,
        "scene_dur": scene_dur,
        "speech_dur": speech_dur,
        "silence_dur": silence_dur,
        "n_silence_windows": n_silence_windows,
        "n_visual_events": n_visual_events,
        "n_tokens": n_tokens,
        "TTS_speech_dur": TTS_speech_dur,
        "fits_silence": fits_silence,
        "human_AD_sim": human_AD_sim,
        "cov_score": cov_score,
        "redun_score": redun_score,
        "quality_score": quality_score,
    })

    return phaseA

phaseA = build_phase_a(stage1)
phaseA.head()

,scene_id,cand_id,scene_dur,speech_dur,silence_dur,n_silence_windows,n_visual_events,n_tokens,TTS_speech_dur,fits_silence,human_AD_sim,cov_score,redun_score,quality_score
0,s0,A,1.862,7.7,-5.838,7.7,5,76,30.4,0,7.7,7.7,7.7,4.204
1,s1,A,2.705,7.7,-4.995,7.7,5,119,47.6,0,7.7,7.7,7.7,4.204
2,s2,A,1.101,7.7,-6.599,7.7,5,37,14.8,0,7.7,7.7,7.7,4.204
3,s3,A,2.281,7.7,-5.419,7.7,5,83,33.2,0,7.7,7.7,7.7,4.204
4,s4,A,1.281,7.7,-6.419,7.7,5,69,27.6,0,7.7,7.7,7.7,4.204


In [ ]:
# i can't figure out how the data is annotated, i'm looking at tvad_anno_context_st 
# and the shots are labeled but i don't know which frames they refer to. i don't think i can extract
# these "PLACEHOLDER" features and need to base it on text alone
# make a response variable, "quality_score" based on how similar the embeddings for the human and AI-generated text are
    # sentence transformers for seeing similarity between text

In [4]:
import pandas as pd
import ast
from sentence_transformers import SentenceTransformer, util

stage2 = pd.read_csv("stage2_llama3_assistant.csv")
anno = pd.read_csv("tvad_anno_context-9-24_face-0.2-0.4_scale_thread.csv")

anno_lookup = (
    anno[["anno_idx", "tvad_name", "AD_start", "AD_end"]]
    .drop_duplicates(subset="anno_idx")
    .set_index("anno_idx")
)

CAND_LABELS = ["A", "B", "C", "D", "E"]

sim_model = SentenceTransformer("all-MiniLM-L6-v2")

def build_phase_a(stage2: pd.DataFrame, anno_lookup: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for _, row in stage2.iterrows():
        candidates = ast.literal_eval(row["text_gen"])
        scene_dur = row["end"] - row["start"]

        idx = row["anno_idx"]
        if idx in anno_lookup.index:
            tvad_name = anno_lookup.loc[idx, "tvad_name"]
            AD_start = anno_lookup.loc[idx, "AD_start"]
            AD_end = anno_lookup.loc[idx, "AD_end"]
        else:
            tvad_name = None
            AD_start = None
            AD_end = None

        for i, cand_text in enumerate(candidates):
            n_tokens = len(str(cand_text).split())
            TTS_speech_dur = n_tokens / 2.5

            rows.append({
                "scene_id": f"s{int(idx)}",
                "cand_id": CAND_LABELS[i],
                "scene_dur": scene_dur,
                "n_tokens": n_tokens,
                "TTS_speech_dur": TTS_speech_dur, # estimate of how long it'll take text-to-speech to say out loud
                "human_AD_text": row["text_gt"],
                "gen_AD_text": cand_text,
                "tvad_name": tvad_name,
                "AD_start": AD_start,
                "AD_end": AD_end,
            })

    df = pd.DataFrame(rows)

    # adding a response variable based on similarity between human & AI-text embeddings
    print('sentence embeddings:')
    human_embs = sim_model.encode(df["human_AD_text"].tolist(), batch_size=64, show_progress_bar=True)
    gen_embs = sim_model.encode(df["gen_AD_text"].tolist(), batch_size=64, show_progress_bar=True)
    df["quality_score"] = util.cos_sim(gen_embs, human_embs).diagonal().tolist()

    return df

phaseA = build_phase_a(stage2, anno_lookup)
print(len(phaseA)) # should be ~15k, 5 candidates for 3k scenes
phaseA.head(10)

/home/mellie/miniconda3/envs/AD_selection/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 792.80it/s]


sentence embeddings:


Batches: 100%|██████████| 234/234 [02:22<00:00,  1.64it/s]


14915


,scene_id,cand_id,scene_dur,n_tokens,TTS_speech_dur,human_AD_text,gen_AD_text,tvad_name,AD_start,AD_end,quality_score
0,s0,A,1.862,7,2.8,His umbrella springs open between them.,"Phoebe holds Rachel's hand, looking at her.",friends_s01e01_seg02_clip_04,240.250,242.112,0.172184
1,s0,B,1.862,8,3.2,His umbrella springs open between them.,"Rachel kneels, holding a cup, as Phoebe gazes.",friends_s01e01_seg02_clip_04,240.250,242.112,0.164244
2,s0,C,1.862,9,3.6,His umbrella springs open between them.,"Phoebe Buffay stands in her wedding dress, hol...",friends_s01e01_seg02_clip_04,240.250,242.112,0.095804
3,s0,D,1.862,8,3.2,His umbrella springs open between them.,"Rachel looks up at Phoebe, cup in hand.",friends_s01e01_seg02_clip_04,240.250,242.112,0.175903
4,s0,E,1.862,9,3.6,His umbrella springs open between them.,"Phoebe Buffay stands, holding Rachel's hand, i...",friends_s01e01_seg02_clip_04,240.250,242.112,0.093899
5,s1,A,2.705,12,4.8,"He awkwardly sits down, and Joey pats his shou...","Rachel stands in a wedding dress, talking to P...",friends_s01e01_seg02_clip_04,242.653,245.358,0.226167
6,s1,B,2.705,11,4.4,"He awkwardly sits down, and Joey pats his shou...","Phoebe sits on the couch, surrounded by Monica...",friends_s01e01_seg02_clip_04,242.653,245.358,0.453110
7,s1,C,2.705,13,5.2,"He awkwardly sits down, and Joey pats his shou...","Rachel wears a wedding dress, while Joey munch...",friends_s01e01_seg02_clip_04,242.653,245.358,0.371190
8,s1,D,2.705,13,5.2,"He awkwardly sits down, and Joey pats his shou...","Monica stands behind Phoebe, as Rachel talks t...",friends_s01e01_seg02_clip_04,242.653,245.358,0.206705
9,s1,E,2.705,17,6.8,"He awkwardly sits down, and Joey pats his shou...","Phoebe sits on the couch, with Ross and Monica...",friends_s01e01_seg02_clip_04,242.653,245.358,0.296691


In [5]:
print(phaseA.head())

  scene_id cand_id  scene_dur  n_tokens  TTS_speech_dur  \
0       s0       A      1.862         7             2.8   
1       s0       B      1.862         8             3.2   
2       s0       C      1.862         9             3.6   
3       s0       D      1.862         8             3.2   
4       s0       E      1.862         9             3.6   

                             human_AD_text  \
0  His umbrella springs open between them.   
1  His umbrella springs open between them.   
2  His umbrella springs open between them.   
3  His umbrella springs open between them.   
4  His umbrella springs open between them.   

                                         gen_AD_text  \
0        Phoebe holds Rachel's hand, looking at her.   
1     Rachel kneels, holding a cup, as Phoebe gazes.   
2  Phoebe Buffay stands in her wedding dress, hol...   
3            Rachel looks up at Phoebe, cup in hand.   
4  Phoebe Buffay stands, holding Rachel's hand, i...   

                      tvad_name

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

def add_consensus_score(df):
    scores = []
    for scene_id, group in df.groupby("scene_id"):
        embs = model.encode(group["gen_AD_text"].tolist())
        sim_matrix = util.cos_sim(embs, embs).numpy()

        np.fill_diagonal(sim_matrix, 0)
        consensus = sim_matrix.mean(axis=1)
        scores.extend(consensus.tolist())
    df["consensus_score"] = scores # how similar the candidate ADs are to each other
    return df

phaseA_t = add_consensus_score(phaseA)
phaseA_t.head()

/home/mellie/miniconda3/envs/AD_selection/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1111.98it/s]


,scene_id,cand_id,scene_dur,n_tokens,human_AD_text,gen_AD_text,quality_score,tvad_name,consensus_score
0,s0,A,1.862,7,His umbrella springs open between them.,"Phoebe holds Rachel's hand, looking at her.",0.172185,friends_s01e01_seg02_clip_04,0.609855
1,s0,B,1.862,8,His umbrella springs open between them.,"Rachel kneels, holding a cup, as Phoebe gazes.",0.164244,friends_s01e01_seg02_clip_04,0.583783
2,s0,C,1.862,9,His umbrella springs open between them.,"Phoebe Buffay stands in her wedding dress, hol...",0.095804,friends_s01e01_seg02_clip_04,0.540911
3,s0,D,1.862,8,His umbrella springs open between them.,"Rachel looks up at Phoebe, cup in hand.",0.175903,friends_s01e01_seg02_clip_04,0.589506
4,s0,E,1.862,9,His umbrella springs open between them.,"Phoebe Buffay stands, holding Rachel's hand, i...",0.093899,friends_s01e01_seg02_clip_04,0.592144
